In [20]:
from pyChimera_main.chimera import *
import codonbias as cb
import pandas as pd
from Bio import SeqIO, Seq
from Bio.SeqRecord import SeqRecord
from pyfaidx import Fasta
import re
from time import time
import pickle
import numpy as np
from itertools import product
from collections import Counter
import logging
from datetime import datetime
import os
import warnings
warnings.filterwarnings("ignore")
from Bio import BiopythonWarning

In [2]:
folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/'

In [3]:
# #tomato
organism_name='tomato'
tomato_folder = folder + 'tomato_data/'
tomato_chrs = Fasta(tomato_folder + "SollycM82_v1.0.fasta")
orfs = Fasta(tomato_folder+"SollycM82_genes_v1.1.1.CDS.fasta")
orfs = pd.DataFrame({'seq':[str(orfs[x]) for x in list(orfs.records)],
                     'id':[x[5:] for x in list(orfs.records)]})
cai = cb.scores.CodonAdaptationIndex(ref_seq=orfs.seq.to_list())
cdss = pd.read_csv(tomato_folder+'tomato_cdss.csv',dtype={'chrm': str})
cdss['transcript_id'] = cdss['description'].str.extract(r'Parent=mRNA:([^;]+)')
with open(tomato_folder+'tomato_codon_freqs.pkl', 'rb') as file:
    codon_freqs = pickle.load(file)
with open(tomato_folder+'tomato_aa_freqs.pkl', 'rb') as file:
    aa_freqs = pickle.load(file)
with open(tomato_folder+'tomato_SA_cod.pkl', 'rb') as file:
    tomato_SA_cod = pickle.load(file)
# tomato = pd.read_csv(tomato_folder + 'processed_tomato.csv')
tomato = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/tomato_data/tomato_roots.csv')
print('done')

In [4]:
# #fly
organism_name='fly'
fly_folder = folder + 'fly_data/'
fly_chrs = Fasta(fly_folder + "GCF_905115235.1_iHerIll2.2.curated.20191125_genomic.fna")
orfs = Fasta(fly_folder+"cds_from_genomic.fna")
orfs = pd.DataFrame({'seq':[str(orfs[x]) for x in list(orfs.records)],
                     'id':[re.findall(r'(?:XP|YP)_\d+\.\d+', x)[0] for x in list(orfs.records)]})
cai = cb.scores.CodonAdaptationIndex(ref_seq=orfs.seq.to_list())
cdss = pd.read_csv(fly_folder+'fly_cdss.csv',dtype={'chrm': str})
cdss['transcript_id'] = cdss['description'].str.extract(r'protein_id=([^;]+)')
with open(fly_folder+'fly_codon_freqs.pkl', 'rb') as file:
    codon_freqs = pickle.load(file)
with open(fly_folder+'fly_aa_freqs.pkl', 'rb') as file:
    aa_freqs = pickle.load(file)
with open(fly_folder+'fly_SA_cod.pkl', 'rb') as file:
    fly_SA_cod = pickle.load(file)
fly = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Fly/fly.csv')
print('done')

In [3]:
#human
organism_name='human'
human_folder = folder + 'human_data/'
human_chrs = Fasta("/tamir2/shaicohen1/EXPosition/code/Data/Chromosome/hg38.fa")
orfs = Fasta(human_folder+"Homo_sapiens.GRCh38.cds.all.fa")
orfs = pd.DataFrame({'seq':[str(orfs[x]) for x in list(orfs.records)], 'id':[x[:x.index('.')] for x in list(orfs.records)]})
cai = cb.scores.CodonAdaptationIndex(ref_seq=orfs.seq.to_list())
cdss = pd.read_csv(human_folder+'human_cdss.csv',dtype={'chrm': str})
cdss['transcript_id'] = cdss['description'].str.extract(r'transcript_id\s+"(ENST\d+)"')
with open(human_folder+'human_codon_freqs.pkl', 'rb') as file:
    codon_freqs = pickle.load(file)
with open(human_folder+'human_aa_freqs.pkl', 'rb') as file:
    aa_freqs = pickle.load(file)
with open(human_folder+'human_SA_cod.pkl', 'rb') as file:
    human_SA_cod = pickle.load(file)
print('done')

done


In [7]:
#shrimp
organism_name='shrimp'
shrimp_folder = folder+'/shrimp_data/'
shrimp_chrs = Fasta(shrimp_folder+'GCF_040412425.1_ASM4041242v1_genomic.fna')
orfs = Fasta(shrimp_folder+"cds_from_genomic.fna")
orfs = pd.DataFrame({'seq':[str(orfs[x]) for x in list(orfs.records)],
                     'id':[re.findall(r'(?:XP|YP)_\d+\.\d+', x)[0] for x in list(orfs.records)]})
cai = cb.scores.CodonAdaptationIndex(ref_seq=orfs.seq.to_list())
cdss = pd.read_csv(shrimp_folder+'shrimp_cdss.csv',dtype={'chrm': str})
cdss['transcript_id'] = cdss['description'].str.extract(r'protein_id=([^;]+)')
with open(shrimp_folder + 'shrimp_codon_freqs.pkl', 'rb') as file:
    codon_freqs = pickle.load(file)
with open(shrimp_folder + 'shrimp_aa_freqs.pkl', 'rb') as file:
    aa_freqs = pickle.load(file)
with open(shrimp_folder+'shrimp_SA_cod.pkl', 'rb') as file:
    shrimp_SA_cod = pickle.load(file)
# shrimp = pd.read_csv(shrimp_folder+'found_shrimp.csv')
shrimp = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/shrimp_data/new_shrimp.csv')

In [8]:
# ######################################
# # # IMPORTANT
# organism_name = 'tomato'
# data_folder = folder + f'{organism_name}_data/'
# ######################################
# # #chimera
# print('start creating suffix array')
# orfs_for_chimera = nt2codon([x for x in orfs.seq.to_list() if len(x)%3==0 and 'N' not in x])
# SA_cod = build_suffix_array(orfs_for_chimera)
# with open(data_folder+f'{organism_name}_SA_cod.pkl','wb') as f:
#     pickle.dump(SA_cod, f)
# print('done making and saving suffix array')
# #####################################
# # # CAI
# print('start creating aa and codon freqs')
# aa_list = ['C', 'S', 'R', 'M', 'V', 'Q', 'K', 'A', 'G', 'I', 'D', 'L', 'N', 'F', 'T', 'E', 'Y', 'W', 'H', 'P']
# nucleotides = ['A', 'T', 'C', 'G']
# all_codons = [''.join(codon) for codon in product(nucleotides, repeat=3)]
# codon_freqs = {x:0 for x in all_codons}
# aa_freqs = {x:0 for x in aa_list}
# for i,seq in enumerate(orfs.seq.to_list()):
#     if i%100==0:
#         print(f'{i/orfs.shape[0]:.3f}')
#     for j in range(0,len(seq),3):
#         codon = seq[j:j+3]
#         if 'N' in codon or len(codon)!=3:
#             continue
#         codon_freqs[codon]+=1
#         aa = str(Seq.Seq(codon).translate())
#         if aa!='*':
#             aa_freqs[aa]+=1
# sum_codons = sum([x for x in codon_freqs.values()])
# sum_aas = sum([x for x in aa_freqs.values()])
# for x in codon_freqs.keys():
#     codon_freqs[x] = codon_freqs[x]/sum_codons
# for x in aa_freqs.keys():
#     aa_freqs[x] = aa_freqs[x]/sum_aas
# with open(data_folder+f'{organism_name}_codon_freqs.pkl','wb') as f:
#     pickle.dump(codon_freqs, f)
# with open(data_folder+f'{organism_name}_aa_freqs.pkl','wb') as f:
#     pickle.dump(aa_freqs, f)
# print('done making and saving aa and codon freqs')

In [9]:
# sites_folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/output_sites/'
# ics = pd.read_csv(sites_folder + 'ICS_w_CAI.csv')
# k562 = pd.read_csv(sites_folder + 'K562_w_CAI.csv')
# u937 = pd.read_csv(sites_folder + 'U937_w_CAI.csv')
# T = pd.read_csv(sites_folder + 'T_w_CAI.csv')
# tomato = pd.read_csv(sites_folder + 'Tomato_w_CAI.csv')
# shrimp = pd.read_csv(sites_folder + 'Shrimp_w_CAI.csv')
# Leenay = pd.read_csv(sites_folder + 'Leenay_w_CAI.csv')
# DeepCRISPR_hek293 = pd.read_csv(sites_folder + 'DeepCRISPR_hek293_w_CAI.csv')
# DeepCRISPR_hela = pd.read_csv(sites_folder + 'DeepCRISPR_hela_w_CAI.csv')
# DeepHF = pd.read_csv(sites_folder + 'DeepHF_w_CAI.csv')

In [10]:
# all_dfs = [ics,k562,u937,T,
#            tomato,shrimp,
#            Leenay,DeepCRISPR_hek293,DeepCRISPR_hela,DeepHF]
# all_dfs_names = ['ICS','K562','U937','T',
#                  'Tomato','Shrimp',
#                  'Leenay','DeepCRISPR_hek293','DeepCRISPR_hela','DeepHF']

In [7]:
def find_overlap_genes(start,stop,chrm):
    start_matches = (cdss.start<=start) & (start<=cdss.stop) & (chrm==cdss.chrm)
    stop_matches = (cdss.start<=stop) & (stop<=cdss.stop) & (chrm==cdss.chrm)
    df_match = cdss[start_matches | stop_matches]
    return df_match

In [8]:
def get_codons(df_match,start,stop,w):
    for i in df_match.index:
        chrm,strand_orf_start,orf_stop = df_match.loc[i,['chrm','strand','start','stop']]
        overlap_start = max(start,df_match)
        # seq = chrs[chrm][]

In [9]:
#from codonbias module
def geomean(seq,aa_or_codon,freqs):
    """
    Compute the geometric mean based on codon scores given in
    `log_weights` (weights in logarithmic scale), and codon counts give
    in `counts`.

    input:
    seq : string (aa or codons)
    aa_or_codons : bool
    freqs : dict of freqs for aa/codons
    
    to be calc (orig parameters)
    ----------
    log_weights : pandas.Series
        Codon scores in logarithmic scale, with codons as index and scores
        as values.
    counts : pandas.Series
        Codon counts, with codons as index and counts as values.

    Returns
    -------
    float
        Geometric mean.
    """
    if aa_or_codon=='aa':
        seq = seq.replace('*', '')
        if len(seq)==0:
            return 0
        curr_idxs = list(set(seq))
        log_weights = pd.Series(np.log([freqs[x] for x in set(seq)]), index=curr_idxs)
        counts = pd.Series(Counter(seq))
    elif aa_or_codon=='codon':
        codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
        curr_idxs = list(set(codons))
        log_weights = pd.Series(np.log([freqs[x] for x in set(codons)]), index=curr_idxs)
        counts = pd.Series(Counter(codons))
    nn = log_weights.index[np.isfinite(log_weights)]
    return np.exp((log_weights[nn] * counts.reindex(nn)).sum() / counts.reindex(nn).sum())

In [10]:
def print_log(x):
    print(x)
    # logging.info(x)

In [14]:
def calc_chimera_CAI_scores(row,organism='human',calc_CAI_chimera='all'):
    # wtf_idxs=[]
    if calc_CAI_chimera=='all':
        num_feats = 24
    elif calc_CAI_chimera=='CAI':
        num_feats = 18
    elif calc_CAI_chimera=='chimera':
        num_feats = 6
    if row.name%50==0:
        print_log(row.name)
    scores = []
    g_rna_info = row['g_rna_info']
    # print_log(g_rna_info)
    if organism=='human':
        split_g_rna_info = g_rna_info.split('_')
        char_to_split = '_'
        if len(split_g_rna_info)!=4 or (split_g_rna_info[0] not in [str(i) for i in range(1,24)] and split_g_rna_info[0] not in ['X','Y']):
            return [[] for _ in range(num_feats)]
        SA_cod = human_SA_cod
    elif organism=='shrimp':
        char_to_split = ';'
        SA_cod = shrimp_SA_cod
    elif organism=='fly':
        char_to_split = ';'
        SA_cod = fly_SA_cod
    elif organism=='tomato':
        char_to_split = ';'
        SA_cod = tomato_SA_cod

    chrm,site_start,site_stop,site_strand = g_rna_info.split(char_to_split)
    chimera_scores = {'':0}
    codon_freqs_scores = {'':0}
    aa_freqs_scores = {'':0}
    cai_scores = {'':0}
    for w in [0,10,20]:
        # print_log('w',w)
        chimera_mean_score = []
        aa_freqs_mean_score = []
        codon_freqs_mean_score = []
        cai_mean_score = []
        cai_max_score = 0
        chimera_max_score = 0
        aa_freqs_max_score = 0
        codon_freqs_max_score = 0
        # cai_max_expr_score = 0
        # aa_freqs_expr_max_score = 0
        # codon_freqs_expr_max_score = 0
        # chimera_max_expr_score = 0
        site_start = int(site_start)-w
        site_stop = int(site_stop)+w
        df_match = find_overlap_genes(site_start,site_stop,chrm)
        for j in df_match.index:
            # if j!=1:
            #     continue
            # print_log('j',j,'w',w)
            cds_start,cds_stop,cds_strand,transcript_id = df_match.loc[j,['start', 'stop', 'strand','transcript_id']]
            curr_orf_match = orfs[orfs['id']==transcript_id]
            if curr_orf_match.shape[0]!=1:
                print('WTF')
                wtf_idxs.append([df_name,row.name,j,'curr_orf_match not 1'])
            orf_seq = curr_orf_match.seq.iloc[0]
            if organism == 'human':
                curr_cds_seq = str(human_chrs['chr'+chrm][cds_start:cds_stop])
            elif organism == 'shrimp':
                curr_cds_seq = str(shrimp_chrs[chrm][cds_start:cds_stop])
            elif organism == 'fly':
                curr_cds_seq = str(fly_chrs[chrm][cds_start:cds_stop])   
            elif organism == 'tomato':
                curr_cds_seq = str(tomato_chrs[chrm][cds_start:cds_stop])  
            curr_cds_coords = list(range(cds_start,cds_stop+1))
            if cds_strand=='-':
                curr_cds_seq = Seq.reverse_complement(curr_cds_seq)
                curr_cds_coords.reverse()
            overlap_start = max(site_start,cds_start)
            overlap_stop = min(site_stop,cds_stop)
            overlap_start_idx = curr_cds_coords.index(overlap_start)
            overlap_stop_idx = curr_cds_coords.index(overlap_stop)
            if overlap_start_idx>overlap_stop_idx:
                overlap_start_idx, overlap_stop_idx = overlap_stop_idx, overlap_start_idx
            overlap_seq = curr_cds_seq[overlap_start_idx:overlap_stop_idx].upper()
            start_overlap_in_orf = orf_seq.index(overlap_seq)
            start_overlap_in_orf = start_overlap_in_orf - (start_overlap_in_orf%3)
            end_overlap_in_orf = start_overlap_in_orf+len(overlap_seq)
            end_overlap_in_orf = end_overlap_in_orf -(end_overlap_in_orf%3)
            codons_seq = orf_seq[start_overlap_in_orf:end_overlap_in_orf]
            codons_seq = ''.join([codons_seq[i_c:i_c+3] for i_c in range(0,len(codons_seq),3) if 'N' not in codons_seq[i_c]])
            # print(len(codons_seq)%3==0,codons_seq)
            if len(codons_seq)==0:
                continue
            aa_seq = str(Seq.Seq(codons_seq).translate())
            if aa_seq not in str(Seq.Seq(orf_seq).translate()):
                print_log('seq not in seq')
                print_log(row.name,codons_seq,len(codons_seq)/3,start_overlap_in_orf%3)
                print_log(str(Seq.Seq(codons_seq).translate()))
                wtf_idxs.append([df_name,row.name,f'j {j}',f'w {w}''all of seq not in orf'])
            elif len(codons_seq)%3!=0:
                print_log('len not divisible by 3')
                wtf_idxs.append([df_name,row.name,f'j {j}',f'w {w}','not divisiable by 3'])
            if codons_seq in chimera_scores.keys():
                chimera_mean_score.append(chimera_scores[codons_seq])         
            else:
                # print_log('calculating...')
                # print_log(f'{df_name}, i={row.name}, j={j}, len={len(codons_seq)}, frame={start_overlap_in_orf%3}')
                # print_log(type(codons_seq),codons_seq)
                # q = str(Seq.Seq(codons_seq).translate())
                # print_log('here')
                
                target_cod = nt2codon(codons_seq)
                curr_chimera_score = calc_cARS(target_cod, SA_cod,verbose=False)
                chimera_mean_score.append(curr_chimera_score)         
                # print(curr_chimera_score)
                chimera_max_score = max(curr_chimera_score,chimera_max_score)
                chimera_scores[codons_seq] = curr_chimera_score
                curr_aa_score = geomean(aa_seq,'aa',aa_freqs)
                curr_codon_score = geomean(codons_seq,'codon',codon_freqs)
                curr_cai_score = cai.get_score(codons_seq)
                codon_freqs_mean_score.append(curr_codon_score)
                aa_freqs_mean_score.append(curr_aa_score)
                cai_mean_score.append(curr_cai_score)
                cai_max_score = max(cai_max_score,curr_cai_score)
                aa_freqs_max_score = max(curr_aa_score,aa_freqs_max_score)
                codon_freqs_max_score = max(curr_codon_score,codon_freqs_max_score)
                aa_freqs_scores[str(Seq.Seq(codons_seq).translate())] = curr_aa_score
                codon_freqs_scores[codons_seq] = curr_codon_score
                cai_scores[codons_seq] = curr_cai_score
            # break
        if cai_mean_score!=[]:
            cai_mean_score = np.mean(cai_mean_score)
        else:
            cai_mean_score=0 
            
        if aa_freqs_mean_score!=[]:
            aa_freqs_mean_score = np.mean(aa_freqs_mean_score)
        else:
            aa_freqs_mean_score=0
            
        if codon_freqs_mean_score!=[]:
            codon_freqs_mean_score = np.mean(codon_freqs_mean_score)
        else:
            codon_freqs_mean_score = 0
            
        if chimera_mean_score!=[]:
            chimera_mean_score = np.mean(chimera_mean_score)
        else:
            chimera_mean_score=0 
                        
        scores.extend([codon_freqs_mean_score,codon_freqs_max_score,
                       aa_freqs_mean_score,aa_freqs_max_score,
                       cai_mean_score, cai_max_score,chimera_mean_score,chimera_max_score])
    #     # break
    # if wtf_idxs:
    #     print_log('wtf', wtf_idxs)
    # if len(scores)!=num_feats:
    #     print(row.name,len(scores),'expected',num_feats)
    return scores

In [24]:
cols = [f'{name}_{f}{m}_{win}'
        for win in [0,10,20] 
        for name,f in zip(['codon','aa','CAI','chimera'],['freqs_','freqs_','','']) 
        for m in ['avg','max']]

In [27]:
df = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/T_w_reads_w2_w8.csv')
df=df.iloc[:10]
df_orig_cub=df[cols].copy()

In [22]:
df = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/T_w_reads_w2_w8.csv')
df=df.iloc[:10]

scores = df.apply(lambda row:
                  calc_chimera_CAI_scores(row, organism='human', calc_CAI_chimera='all'),
                  axis=1, result_type='expand')
scores.columns = cols
df = pd.concat([df, scores], axis=1)
print('done')

0
done


In [16]:
from time import time
t=time()
wtf_idxs  = []
cols = [f'{name}_{f}{m}_{win}'
        for win in [0,10,20] 
        for name,f in zip(['codon','aa','CAI','chimera'],['freqs_','freqs_','','']) 
        for m in ['avg','max']]
all_dfs = [tomato]
all_dfs_names = ['tomato_roots']
warnings.filterwarnings("ignore", category=BiopythonWarning, message=".*Partial codon.*")
for i_df in range(len(all_dfs)):
    df = all_dfs[i_df]
    df_name = all_dfs_names[i_df]
        # len(split_g_rna_info)!=4 or (split_g_rna_info[0] not in [str(i) for i in range(1,24)] and split_g_rna_info[0] not in ['X','Y']):
        # return scores
    print_log(df_name)
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    file_name = f'/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/chimera_{df_name}.log' 
    with open(file_name, 'w') as f:
        f.write("------------------")
    logging.basicConfig(filename=file_name, level=logging.INFO)
    print_log('----------------------------------------------------')
    print_log('start')
    print_log(str(datetime.now()))
    print_log(df_name)
    scores = df.apply(lambda row:
                      calc_chimera_CAI_scores(row, organism=organism_name, calc_CAI_chimera='all'),
                      axis=1, result_type='expand')
    scores.columns = cols
    df = pd.concat([df, scores], axis=1)
    df.to_csv(f'/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/cb_chimera_code/output_sites/chimera_sites/corrected_sites/{df_name}_w_CAI_chimera.csv',index=False)
    print_log('done')
    print_log(time()-t)
    logging.shutdown()
    # break

print('ALL DONE')

# ['codon_freqs_avg_0', 'codon_freqs_max_0', 'aa_freqs_avg_0', 'aa_freqs_max_0', 'CAI_freqs_avg_0', 'CAI_freqs_max_0', 'codon_freqs_avg_10', 'codon_freqs_max_10', 'aa_freqs_avg_10', 'aa_freqs_max_10', 'CAI_freqs_avg_10', 'CAI_freqs_max_10', 'codon_freqs_avg_20', 'codon_freqs_max_20', 'aa_freqs_avg_20', 'aa_freqs_max_20', 'CAI_freqs_avg_20', 'CAI_freqs_max_20']

tomato_roots
----------------------------------------------------
start
2025-03-12 10:59:38.671973
tomato_roots
0
50
100
done
7.688435316085815
ALL DONE


In [15]:
transcript_id

'ID=cds-XP_037921499.1;Parent=rna-XM_038065571.1;Dbxref=GeneID:119658279,Genbank:XP_037921499.1;Name=XP_037921499.1;gbkey=CDS;gene=LOC119658279;product=protein yellow;protein_id=XP_037921499.1'

In [25]:
df_match.columns

Index(['chrm', 'type', 'start', 'stop', 'strand', 'frame', 'description',
       'transcript_id'],
      dtype='object')

In [16]:
df_match.columns

Index(['chrm', 'type', 'start', 'stop', 'strand', 'frame', 'description',
       'transcript_id'],
      dtype='object')

In [33]:
row=fly.loc[0]
organism='fly'
calc_CAI_chimera='all'
if calc_CAI_chimera=='all':
    num_feats = 24
elif calc_CAI_chimera=='CAI':
    num_feats = 18
elif calc_CAI_chimera=='chimera':
    num_feats = 6
if row.name%50==0:
    print_log(row.name)
scores = []
g_rna_info = row['g_rna_info']
print(g_rna_info,organism)
if organism=='human':
    print('here')
    split_g_rna_info = g_rna_info.split('_')
    char_to_split = '_'
    if len(split_g_rna_info)!=4 or (split_g_rna_info[0] not in [str(i) for i in range(1,24)] and split_g_rna_info[0] not in ['X','Y']):
        print( [[] for _ in range(num_feats)])
    SA_cod = human_SA_cod
elif organism=='shrimp':
    char_to_split = ';'
    SA_cod = shrimp_SA_cod
elif organism=='fly':
    char_to_split = ';'
    SA_cod = fly_SA_cod

chrm,site_start,site_stop,site_strand = g_rna_info.split(char_to_split)
chimera_scores = {'':0}
codon_freqs_scores = {'':0}
aa_freqs_scores = {'':0}
cai_scores = {'':0}
for w in [0,10,20]:
    # print_log('w',w)
    chimera_mean_score = []
    aa_freqs_mean_score = []
    codon_freqs_mean_score = []
    cai_mean_score = []
    cai_max_score = 0
    chimera_max_score = 0
    aa_freqs_max_score = 0
    codon_freqs_max_score = 0
    # cai_max_expr_score = 0
    # aa_freqs_expr_max_score = 0
    # codon_freqs_expr_max_score = 0
    # chimera_max_expr_score = 0
    site_start = int(site_start)-w
    site_stop = int(site_stop)+w
    df_match = find_overlap_genes(site_start,site_stop,chrm)
    for j in df_match.index:
        # if j!=1:
        #     continue
        # print_log('j',j,'w',w)
        cds_start,cds_stop,cds_strand,frame,transcript_id = df_match.loc[j,['start', 'stop', 'strand', 'frame','transcript_id']]
        curr_orf_match = orfs[orfs['id']==transcript_id]
        if curr_orf_match.shape[0]!=1:
            print('WTF')
            wtf_idxs.append([df_name,row.name,j,'curr_orf_match not 1'])
        orf_seq = curr_orf_match.seq.iloc[0]
        if organism == 'human':
            curr_cds_seq = str(human_chrs['chr'+chrm][cds_start:cds_stop])
        elif organism == 'shrimp':
            curr_cds_seq = str(shrimp_chrs[chrm][cds_start:cds_stop])
        elif organism == 'fly':
            curr_cds_seq = str(fly_chrs[chrm][cds_start:cds_stop])        
        curr_cds_coords = list(range(cds_start,cds_stop+1))
        if cds_strand=='-':
            curr_cds_seq = Seq.reverse_complement(curr_cds_seq)
            curr_cds_coords.reverse()
        overlap_start = max(site_start,cds_start)
        overlap_stop = min(site_stop,cds_stop)
        overlap_start_idx = curr_cds_coords.index(overlap_start)
        overlap_stop_idx = curr_cds_coords.index(overlap_stop)
        if overlap_start_idx>overlap_stop_idx:
            overlap_start_idx, overlap_stop_idx = overlap_stop_idx, overlap_start_idx
        overlap_seq = curr_cds_seq[overlap_start_idx:overlap_stop_idx].upper()
        start_overlap_in_orf = orf_seq.index(overlap_seq)
        start_overlap_in_orf = start_overlap_in_orf - (start_overlap_in_orf%3)
        end_overlap_in_orf = start_overlap_in_orf+len(overlap_seq)
        end_overlap_in_orf = end_overlap_in_orf -(end_overlap_in_orf%3)
        codons_seq = orf_seq[start_overlap_in_orf:end_overlap_in_orf]
        codons_seq = ''.join([codons_seq[i_c:i_c+3] for i_c in range(0,len(codons_seq),3) if 'N' not in codons_seq[i_c]])
        # print(len(codons_seq)%3==0,codons_seq)
        if len(codons_seq)==0:
            continue
        aa_seq = str(Seq.Seq(codons_seq).translate())
        if aa_seq not in str(Seq.Seq(orf_seq).translate()):
            print_log('seq not in seq')
            print_log(row.name,codons_seq,len(codons_seq)/3,start_overlap_in_orf%3)
            print_log(str(Seq.Seq(codons_seq).translate()))
            wtf_idxs.append([df_name,row.name,f'j {j}',f'w {w}''all of seq not in orf'])
        elif len(codons_seq)%3!=0:
            print_log('len not divisible by 3')
            wtf_idxs.append([df_name,row.name,f'j {j}',f'w {w}','not divisiable by 3'])
        if codons_seq in chimera_scores.keys():
            chimera_mean_score.append(chimera_scores[codons_seq])         
        else:
            # print_log('calculating...')
            # print_log(f'{df_name}, i={row.name}, j={j}, len={len(codons_seq)}, frame={start_overlap_in_orf%3}')
            # print_log(type(codons_seq),codons_seq)
            # q = str(Seq.Seq(codons_seq).translate())
            # print_log('here')
            
            target_cod = nt2codon(codons_seq)
            curr_chimera_score = calc_cARS(target_cod, SA_cod,verbose=False)
            chimera_mean_score.append(curr_chimera_score)         
            # print(curr_chimera_score)
            chimera_max_score = max(curr_chimera_score,chimera_max_score)
            chimera_scores[codons_seq] = curr_chimera_score
            curr_aa_score = geomean(aa_seq,'aa',aa_freqs)
            curr_codon_score = geomean(codons_seq,'codon',codon_freqs)
            curr_cai_score = cai.get_score(codons_seq)
            codon_freqs_mean_score.append(curr_codon_score)
            aa_freqs_mean_score.append(curr_aa_score)
            cai_mean_score.append(curr_cai_score)
            cai_max_score = max(cai_max_score,curr_cai_score)
            aa_freqs_max_score = max(curr_aa_score,aa_freqs_max_score)
            codon_freqs_max_score = max(curr_codon_score,codon_freqs_max_score)
            aa_freqs_scores[str(Seq.Seq(codons_seq).translate())] = curr_aa_score
            codon_freqs_scores[codons_seq] = curr_codon_score
            cai_scores[codons_seq] = curr_cai_score
        # break
    if cai_mean_score!=[]:
        cai_mean_score = np.mean(cai_mean_score)
    else:
        cai_mean_score=0 
        
    if aa_freqs_mean_score!=[]:
        aa_freqs_mean_score = np.mean(aa_freqs_mean_score)
    else:
        aa_freqs_mean_score=0
        
    if codon_freqs_mean_score!=[]:
        codon_freqs_mean_score = np.mean(codon_freqs_mean_score)
    else:
        codon_freqs_mean_score = 0
        
    if chimera_mean_score!=[]:
        chimera_mean_score = np.mean(chimera_mean_score)
    else:
        chimera_mean_score=0 
                    
    scores.extend([codon_freqs_mean_score,codon_freqs_max_score,
                   aa_freqs_mean_score,aa_freqs_max_score,
                   cai_mean_score, cai_max_score,chimera_mean_score,chimera_max_score])
    # break
# if wtf_idxs:
#     print_log('wtf', wtf_idxs)
if len(scores)!=num_feats:
    print(row.name,len(scores),'expected',num_feats)
print(scores)

0
NC_051853.1;88374240;88374260;- fly
[np.float64(0.012197005944713419), np.float64(0.012197005944713419), np.float64(0.04053626100117221), np.float64(0.04053626100117221), np.float64(0.7097962257096984), np.float64(0.7097962257096984), np.float64(3.5), np.float64(3.5), np.float64(0.012570724980622072), np.float64(0.012570724980622072), np.float64(0.04352346617962154), np.float64(0.04352346617962154), np.float64(0.6871495661160261), np.float64(0.6871495661160261), np.float64(5.0), np.float64(5.0), np.float64(0.013078992583337444), np.float64(0.013078992583337444), np.float64(0.04100235784004444), np.float64(0.04100235784004444), np.float64(0.6725714047380211), np.float64(0.6725714047380211), np.float64(8.5), np.float64(8.5)]
